<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-30</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>PEFT와 LoRA 원리(PEFT & LoRA Principles)</strong>에 대해 학습합니다.</br></br>
>파라미터 효율적 파인튜닝의 개념과 LoRA의 수학적 원리를 학습해봅시다.

</br>

# PEFT (Parameter-Efficient Fine-Tuning)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">전체 파라미터가 아닌 소수의 파라미터만 학습</mark>하여 효율적으로 파인튜닝하는 기법입니다.

> 전체 파인튜닝(Full Fine-tuning)은 매우 비효율적입니다.</br></br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">7B(70억 파라미터) 모델</mark>을 예로 들면, FP16 기준으로 파라미터 저장에만 **약 14GB GPU 메모리**가 필요하고, 학습 시에는 그래디언트와 옵티마이저 상태가 추가되어 **실제 메모리는 2~3배(28~42GB)** 더 필요합니다.</br></br>
> 대부분의 연구자와 개발자는 이런 규모의 GPU를 보유하지 않으며, 클라우드 비용도 매우 높습니다.
> PEFT는 이러한 문제에서 출발합니다.</br>
> 전체 파라미터 중 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">소수(0.1~1%)만 학습</mark>하더라도 전체 파인튜닝에 근접한 성능을 낼 수 있다는 실험적 발견이 핵심 아이디어입니다. 이 내용을 이해하기 위해서는 파인튜닝(사전학습 모델의 추가 학습), 행렬 분해(큰 행렬을 작은 행렬의 곱으로 근사), 학습률, GPU 메모리(VRAM) 등의 개념에 대한 기본 이해가 필요합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">방식</th>
      <th style="text-align:center">학습 파라미터</th>
      <th style="text-align:center">메모리</th>
      <th style="text-align:center">성능</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Full Fine-tuning</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">100%</mark></td><td style="text-align:center">매우 높음</td><td style="text-align:center">최고</td></tr>
    <tr><td style="text-align:center">PEFT (LoRA)</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">0.1~1%</mark></td><td style="text-align:center">낮음</td><td style="text-align:center">Full에 근접</td></tr>
    <tr><td style="text-align:center">Prompt Tuning</td><td style="text-align:center">프롬프트만</td><td style="text-align:center">최소</td><td style="text-align:center">제한적</td></tr>
  </tbody>
</table>

</br>

# LoRA (Low-Rank Adaptation)
> 가중치 행렬의 변화량을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">저랭크 행렬 분해(Low-Rank Decomposition)</mark>로 근사합니다.

</br>

## 수식: W' = W + BA
> 원래 가중치 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">W는 동결</mark>하고, 작은 행렬 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">B × A만 학습</mark>합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">기호</th>
      <th style="text-align:center">크기</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">W</td><td style="text-align:center">d × d</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">원래 가중치</mark> (동결)</td></tr>
    <tr><td style="text-align:center">A</td><td style="text-align:center">d × r</td><td>다운 프로젝션</td></tr>
    <tr><td style="text-align:center">B</td><td style="text-align:center">r × d</td><td>업 프로젝션</td></tr>
    <tr><td style="text-align:center">r</td><td style="text-align:center">스칼라</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">랭크</mark> (4, 8, 16 등)</td></tr>
  </tbody>
</table>

<div style="text-align:center">

</div>

In [ ]:
# TODO 1: d=4096, r=8로 설정하고 원래 파라미터 수(d*d), LoRA 파라미터 수(d*r + r*d), 비율(ratio)을 계산하여 출력해봅시다.


</br>

## LoRA 핵심 하이퍼파라미터

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
      <th style="text-align:center">권장값</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>r</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">랭크</mark> (표현력)</td><td style="text-align:center">8~16</td></tr>
    <tr><td style="text-align:center"><code>lora_alpha</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">스케일링 계수</mark></td><td style="text-align:center">r의 1~2배</td></tr>
    <tr><td style="text-align:center"><code>lora_dropout</code></td><td>드롭아웃 비율</td><td style="text-align:center">0.05~0.1</td></tr>
    <tr><td style="text-align:center"><code>target_modules</code></td><td>적용 대상 레이어</td><td style="text-align:center">q_proj, v_proj</td></tr>
  </tbody>
</table>

<div style="text-align:center">

</div>

In [ ]:
# TODO 2: LoRA 설정을 구성하여 r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"을 설정하고 결과를 출력해봅시다.


<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">설정</th>
      <th style="text-align:center">값</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">랭크 (r)</td><td style="text-align:center">16</td></tr>
    <tr><td style="text-align:center">알파</td><td style="text-align:center">32</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">스케일링 (alpha/r)</mark></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">2.0</mark></td></tr>
    <tr><td style="text-align:center">대상 모듈</td><td style="text-align:center">q_proj, v_proj</td></tr>
    <tr><td style="text-align:center">태스크 타입</td><td style="text-align:center">CAUSAL_LM</td></tr>
  </tbody>
</table>

💡lora_alpha / r = 스케일링
> 실제 적용 시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">lora_alpha / r</mark>이 LoRA 기여도의 스케일링 계수입니다.</br>
> alpha=32, r=16이면 스케일링=2로, LoRA의 영향이 2배가 됩니다.

💡왜 Low-Rank가 효과적인가?
> 대형 모델의 가중치 변화량(ΔW)은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">본질적으로 저랭크</mark>인 경우가 많습니다.</br>
> 즉, 파인튜닝에 필요한 정보는 전체 차원보다 훨씬 적은 차원으로 표현 가능합니다.

</br>

# Full Fine-tuning의 메모리 문제
> 모델 크기가 커질수록 Full Fine-tuning에 필요한 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">GPU 메모리(VRAM)</mark>가 기하급수적으로 증가합니다.

> Full Fine-tuning은 모든 파라미터를 업데이트하므로, 모델 가중치 외에도 **Gradient**, **Optimizer States(momentum, variance)**, **Activations**를 GPU 메모리에 저장해야 합니다.</br></br>
> 비트 정밀도(Bit Precision)에 따라 파라미터 하나당 차지하는 메모리가 달라지며, 이것이 QLoRA에서 4-bit 양자화를 사용하는 핵심 이유입니다.

## 비트 정밀도별 메모리 비교 (7B 모델 기준)

<div style="text-align:center">

</div>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">정밀도</th>
      <th style="text-align:center">비트 수</th>
      <th style="text-align:center">파라미터당 메모리</th>
      <th style="text-align:center">7B 모델 크기</th>
      <th style="text-align:center">16GB로 학습 가능?</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">FP32</td><td style="text-align:center">32-bit</td><td style="text-align:center">4 bytes</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">28.0 GB</mark></td><td style="text-align:center">불가능</td></tr>
    <tr><td style="text-align:center">FP16/BF16</td><td style="text-align:center">16-bit</td><td style="text-align:center">2 bytes</td><td style="text-align:center">14.0 GB</td><td style="text-align:center">추론만 가능</td></tr>
    <tr><td style="text-align:center">INT8</td><td style="text-align:center">8-bit</td><td style="text-align:center">1 byte</td><td style="text-align:center">7.0 GB</td><td style="text-align:center">추론만 가능</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">NF4</mark></td><td style="text-align:center">4-bit</td><td style="text-align:center">0.5 bytes</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">3.5 GB</mark></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">QLoRA 학습 가능</mark></td></tr>
  </tbody>
</table>

</br>

# QLoRA (Quantized LoRA)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">4-bit 양자화(NF4)</mark>된 Base Model에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LoRA Adapter</mark>를 결합하여, 제한된 GPU 메모리에서도 대형 모델을 학습할 수 있게 하는 기법입니다.

> QLoRA의 핵심은 **"Q(Quantized) + LoRA"**입니다.</br></br>
> Base Model의 가중치를 NF4(4-bit NormalFloat)로 양자화하여 메모리를 대폭 줄이고, 실제 학습은 FP16/BF16의 LoRA Adapter에서만 수행합니다. 이렇게 하면 7B 모델도 16GB VRAM에서 파인튜닝할 수 있습니다.

## QLoRA의 핵심 기술 3가지

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">기술</th>
      <th>설명</th>
      <th style="text-align:center">효과</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">NF4 양자화</mark></td><td>가중치 분포에 최적화된 4-bit 양자화. 일반 INT4보다 정보 손실이 적음</td><td style="text-align:center">메모리 87.5% 절감</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Double Quantization</mark></td><td>양자화 상수(quantization constants)까지 다시 양자화</td><td style="text-align:center">약 0.5GB 추가 절약</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Paged Optimizers</mark></td><td>GPU 메모리 부족 시 CPU 메모리로 자동 스왑</td><td style="text-align:center">OOM 방지</td></tr>
  </tbody>
</table>

## LoRA vs QLoRA 비교

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구분</th>
      <th style="text-align:center">LoRA</th>
      <th style="text-align:center">QLoRA</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Base Model 정밀도</td><td style="text-align:center">FP16/BF16</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">NF4 (4-bit)</mark></td></tr>
    <tr><td style="text-align:center">LoRA Adapter 정밀도</td><td style="text-align:center">FP16/BF16</td><td style="text-align:center">FP16/BF16 (동일)</td></tr>
    <tr><td style="text-align:center">7B 모델 VRAM</td><td style="text-align:center">~14 GB + α</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">~4 GB + α</mark></td></tr>
    <tr><td style="text-align:center">16GB GPU 학습</td><td style="text-align:center">7B 불가</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">7B 가능</mark></td></tr>
    <tr><td style="text-align:center">성능 차이</td><td style="text-align:center">기준</td><td style="text-align:center">1~3% 손실 (무시 가능)</td></tr>
  </tbody>
</table>

💡NF4(4-bit NormalFloat)란?
> 일반적인 INT4 양자화는 균일 간격으로 16단계를 나누지만, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">NF4는 가중치의 실제 분포(정규분포)</mark>에 맞춰 비균일 간격으로 양자화합니다.</br>
> 이로 인해 같은 4-bit라도 정보 손실이 최소화되어, Full Precision 대비 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">성능 저하가 1~3% 수준</mark>에 불과합니다.

💡QLoRA 실전 요약
> Base Model은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">NF4로 동결(freeze)</mark>하고, LoRA Adapter는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">FP16/BF16으로 학습</mark>합니다.</br>
> Unsloth에서 `bnb-4bit` 모델을 로드하면 QLoRA의 "Q" 부분이 자동 적용됩니다.